# 🎓 ACA N° 1 – Aprendizaje Automático
## Proyecto 2: Educación · *Predict Students' Dropout and Academic Success* (UCI id=697)
**CUN · Especialización en Inteligencia Artificial · Docente: Dr. Hamilton Rivera Flor**

Este cuaderno es el **borrador completo** de los Pasos 1 a 4 de la ruta de aprendizaje (exploración, limpieza, modelado clásico y evaluación), sin la parte de Streamlit:

| Paso | Contenido |
|---|---|
| **1. Exploración y EDA** | Estructura, tipos, nulos, duplicados, estadística descriptiva y desviaciones, histogramas, boxplots, outliers, categóricas, relación con el target y correlaciones |
| **2. Limpieza y preprocesamiento** | Estandarización de nombres, duplicados, nulos, tipos, consistencia, outliers (IQR + winsorización), codificación del target, validación y exportación |
| **3. Preparación para modelado** | Partición estratificada 80/20, validación cruzada, pesos de clase y pipelines sin fuga de datos |
| **4. Modelado y evaluación** | Regresión Logística, Random Forest y Gradient Boosting: CV, ajuste de hiperparámetros, test, matrices de confusión, importancia de variables y alerta temprana |
| **5. Conclusiones** | Hallazgos, limitaciones, ética y trabajo futuro (el despliegue en Streamlit queda como siguiente entregable) |

> **Pregunta orientadora:** ¿Cómo transformar datos educativos en conocimiento predictivo para anticipar la deserción de un estudiante?

**Clases del target:** `Dropout` (desertó) · `Enrolled` (sigue matriculado) · `Graduate` (graduado)

## 0. Configuración del entorno y carga de datos

In [ ]:
!pip install ucimlrepo -q   # solo necesario si no usas data.csv

In [ ]:
import re
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (10, 5)
SEED = 42

In [ ]:
# Fuente de datos: 1) data.csv (archivo del curso, separado por ';')  2) si no está, descarga desde UCI (id = 697)
import os

RUTA_CSV = "data.csv"   # en Colab: sube el archivo desde el panel izquierdo (Archivos) o con files.upload()

if os.path.exists(RUTA_CSV):
    df_raw = pd.read_csv(RUTA_CSV, sep=";", encoding="utf-8-sig")
    print(f"✅ Cargado desde {RUTA_CSV}")
else:
    print(f"⚠️ No se encontró {RUTA_CSV}; descargando desde UCI con ucimlrepo...")
    from ucimlrepo import fetch_ucirepo
    predict_students_dropout_and_academic_success = fetch_ucirepo(id=697)

    # data (como DataFrames de pandas)
    X = predict_students_dropout_and_academic_success.data.features
    y = predict_students_dropout_and_academic_success.data.targets
    df_raw = pd.concat([X, y], axis=1)

    # metadatos e información de variables
    print(predict_students_dropout_and_academic_success.metadata)
    display(predict_students_dropout_and_academic_success.variables)

print(f"df_raw -> {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
df_raw.head()

---
# 📊 PARTE 1 — Exploración y EDA

## 1.1 Normalización de nombres de columnas
Los nombres originales tienen espacios, mayúsculas, paréntesis, apóstrofes, una errata (`Nacionality`) e incluso un **tab** oculto en `Daytime/evening attendance`.
Los llevamos a `snake_case` para trabajar sin errores. Es un cambio **solo de etiqueta**, no altera los datos.

In [ ]:
def normalizar_nombre(col):
    """'Mother's qualification' -> 'mothers_qualification'"""
    col = str(col).strip().lower().replace("'", "")
    return re.sub(r"[^a-z0-9]+", "_", col).strip("_")


df = df_raw.copy()
df.columns = [normalizar_nombre(c) for c in df.columns]
df = df.rename(columns={"nacionality": "nationality"})   # corregimos la errata del dataset

TARGET = "target"
FEATURES = [c for c in df.columns if c != TARGET]

# Clasificación de variables según el diccionario de datos
BINARY_COLS = ["daytime_evening_attendance", "displaced", "educational_special_needs", "debtor",
               "tuition_fees_up_to_date", "gender", "scholarship_holder", "international"]
CAT_COLS = ["marital_status", "application_mode", "application_order", "course", "previous_qualification",
            "nationality", "mothers_qualification", "fathers_qualification",
            "mothers_occupation", "fathers_occupation"]
NUM_COLS = [c for c in FEATURES if c not in BINARY_COLS + CAT_COLS]

faltan = set(BINARY_COLS + CAT_COLS) - set(df.columns)
assert not faltan, f"Columnas esperadas que no existen: {faltan}"

print(f"Binarias ({len(BINARY_COLS)}), categóricas codificadas ({len(CAT_COLS)}), numéricas continuas/conteos ({len(NUM_COLS)})")
print("\nNuméricas:", NUM_COLS)

## 1.2 Estructura general del dataset

In [ ]:
print(f"Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1024:,.1f} KB\n")
df.info()

In [ ]:
display(df.head(10))
display(df.sample(5, random_state=SEED))

## 1.3 Valores nulos (y valores 'disfrazados' de nulo)

In [ ]:
nulos = pd.DataFrame({"nulos": df.isna().sum(), "pct_nulos": (df.isna().mean() * 100).round(2)})
total_nulos = int(nulos["nulos"].sum())
print(f"Total de valores nulos en el dataset: {total_nulos}")

if total_nulos > 0:
    display(nulos[nulos["nulos"] > 0].sort_values("nulos", ascending=False))
    nulos[nulos["nulos"] > 0]["pct_nulos"].sort_values().plot.barh(color="tomato", figsize=(8, 4))
    plt.title("% de nulos por columna"); plt.xlabel("%"); plt.show()
else:
    print("✅ No hay valores NaN en ninguna columna.")

# Marcadores de nulo escritos como texto en columnas tipo objeto
for c in df.select_dtypes(include="object").columns:
    vacios = df[c].astype(str).str.strip().isin(["", "?", "NA", "N/A", "null", "None", "nan"]).sum()
    print(f"Columna '{c}': {vacios} valores vacíos/marcadores de nulo")

## 1.4 Registros duplicados

In [ ]:
n_dup = int(df.duplicated().sum())
print(f"Filas completamente duplicadas: {n_dup} ({n_dup / len(df) * 100:.2f}%)")
if n_dup > 0:
    display(df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(10))
else:
    print("✅ No hay filas duplicadas.")
print("\nNota: este dataset no trae un ID de estudiante, por eso solo podemos detectar duplicados exactos de fila.")

## 1.5 Tipos de variables y cardinalidad

In [ ]:
card = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "valores_unicos": df.nunique(),
    "min": df[FEATURES].min(),
    "max": df[FEATURES].max(),
})
card["tipo_asignado"] = np.where(card.index.isin(BINARY_COLS), "binaria",
                          np.where(card.index.isin(CAT_COLS), "categórica (código)",
                          np.where(card.index == TARGET, "target", "numérica")))
display(card.sort_values("tipo_asignado"))

## 1.6 Estadística descriptiva y **desviaciones**
Además de `describe()`, calculamos varianza, **coeficiente de variación** (dispersión relativa: `std / media`), asimetría, curtosis y un test de normalidad (D'Agostino–Pearson).
- |asimetría| > 1 → distribución muy sesgada.
- CV > 100 % → mucha dispersión respecto a la media.
- p < 0.05 → se rechaza normalidad.

In [ ]:
desc = df[NUM_COLS].describe().T
desc["varianza"] = df[NUM_COLS].var()
desc["cv_%"] = (desc["std"] / desc["mean"].replace(0, np.nan)) * 100
desc["rango_iqr"] = desc["75%"] - desc["25%"]
desc["asimetria"] = df[NUM_COLS].skew()
desc["curtosis"] = df[NUM_COLS].kurt()
desc["p_normalidad"] = [stats.normaltest(df[c].dropna())[1] for c in NUM_COLS]
display(desc)

print("\nVariables muy sesgadas (|asimetría| > 1):")
print(desc.loc[desc["asimetria"].abs() > 1, "asimetria"].sort_values().round(2).to_string())
print("\nVariables con alta dispersión relativa (CV > 100%):")
print(desc.loc[desc["cv_%"] > 100, "cv_%"].sort_values(ascending=False).round(1).to_string())

In [ ]:
# Desviación estándar comparativa (en escala original y normalizada por la media)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
desc["std"].sort_values().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Desviación estándar (escala original)")
desc["cv_%"].sort_values().plot.barh(ax=axes[1], color="darkorange")
axes[1].set_title("Coeficiente de variación (%)")
plt.tight_layout(); plt.show()

## 1.7 Variable objetivo (`target`)

In [ ]:
conteo = df[TARGET].value_counts()
resumen_target = pd.DataFrame({"estudiantes": conteo, "porcentaje": (df[TARGET].value_counts(normalize=True) * 100).round(2)})
display(resumen_target)
print(f"Razón de desbalance (mayoritaria / minoritaria): {conteo.max() / conteo.min():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.countplot(data=df, x=TARGET, order=conteo.index, ax=axes[0])
axes[0].bar_label(axes[0].containers[0]); axes[0].set_title("Distribución del target")
axes[1].pie(conteo, labels=conteo.index, autopct="%1.1f%%", startangle=90)
axes[1].set_title("Proporción por clase")
plt.tight_layout(); plt.show()

## 1.8 Histogramas de las variables numéricas

In [ ]:
def crear_grilla(n, ncols=4, w=5, h=3.6):
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * w, nrows * h))
    axes = np.array(axes).ravel()
    for ax in axes[n:]:
        ax.axis("off")
    return fig, axes


fig, axes = crear_grilla(len(NUM_COLS))
for ax, c in zip(axes, NUM_COLS):
    sns.histplot(df[c], kde=True, ax=ax, color="steelblue", bins=30)
    ax.axvline(df[c].mean(), color="red", ls="--", lw=1, label="media")
    ax.axvline(df[c].median(), color="green", ls="-", lw=1, label="mediana")
    ax.set_title(c, fontsize=9); ax.set_xlabel(""); ax.legend(fontsize=7)
plt.suptitle("Histogramas + KDE (rojo = media, verde = mediana)", y=1.002, fontsize=14)
plt.tight_layout(); plt.show()

## 1.9 Boxplots y detección de outliers (regla IQR)

In [ ]:
fig, axes = crear_grilla(len(NUM_COLS))
for ax, c in zip(axes, NUM_COLS):
    sns.boxplot(x=df[c], ax=ax, color="lightcoral")
    ax.set_title(c, fontsize=9); ax.set_xlabel("")
plt.tight_layout(); plt.show()

In [ ]:
def tabla_outliers(data, cols, k=1.5):
    filas = []
    for c in cols:
        q1, q3 = data[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        li, ls = q1 - k * iqr, q3 + k * iqr
        n = int(((data[c] < li) | (data[c] > ls)).sum())
        filas.append({"variable": c, "Q1": q1, "Q3": q3, "IQR": iqr, "lim_inf": li, "lim_sup": ls,
                      "n_outliers": n, "pct_outliers": round(n / len(data) * 100, 2)})
    return pd.DataFrame(filas).set_index("variable").sort_values("pct_outliers", ascending=False)


display(tabla_outliers(df, NUM_COLS))

## 1.10 Variables binarias y categóricas

In [ ]:
# Binarias: distribución y composición del target dentro de cada categoría (% por fila)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, c in zip(axes.ravel(), BINARY_COLS):
    ct = pd.crosstab(df[c], df[TARGET], normalize="index") * 100
    ct.plot(kind="bar", stacked=True, ax=ax, rot=0, legend=False)
    n = df[c].value_counts().sort_index()
    ax.set_xticklabels([f"{i}\n(n={n[i]:,})" for i in ct.index])
    ax.set_title(c, fontsize=10); ax.set_xlabel(""); ax.set_ylabel("% del target")
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, title="Target")
plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.show()

In [ ]:
# Cardinalidad de las categóricas codificadas y frecuencia de sus categorías principales
for c in CAT_COLS:
    vc = df[c].value_counts()
    print(f"{c:<26} {vc.size:>3} categorías | top-3: " + ", ".join(f"{k} ({v / len(df) * 100:.1f}%)" for k, v in vc.head(3).items()))

In [ ]:
# Tasa de deserción por programa académico (course)
g = df.groupby("course")[TARGET].agg(
    n="count",
    tasa_desercion=lambda s: (s == "Dropout").mean() * 100,
    tasa_graduacion=lambda s: (s == "Graduate").mean() * 100,
).sort_values("tasa_desercion", ascending=False)
display(g)

g["tasa_desercion"].sort_values().plot.barh(figsize=(10, 6), color="tomato")
plt.title("Tasa de deserción (%) por código de curso"); plt.xlabel("%"); plt.show()

## 1.11 Variables numéricas vs. target

In [ ]:
CLAVES = ["admission_grade", "previous_qualification_grade", "age_at_enrollment",
          "curricular_units_1st_sem_approved", "curricular_units_1st_sem_grade",
          "curricular_units_2nd_sem_approved", "curricular_units_2nd_sem_grade",
          "curricular_units_2nd_sem_enrolled"]
CLAVES = [c for c in CLAVES if c in df.columns]
orden = ["Dropout", "Enrolled", "Graduate"]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, c in zip(axes.ravel(), CLAVES):
    sns.boxplot(data=df, x=TARGET, y=c, order=orden, ax=ax)
    ax.set_title(c, fontsize=10); ax.set_xlabel("")
plt.tight_layout(); plt.show()

display(df.groupby(TARGET)[CLAVES].agg(["mean", "std"]).round(2).T)

## 1.12 Correlaciones
- **Pearson** entre variables numéricas → detecta multicolinealidad.
- Correlación de cada variable (numéricas + binarias) con `Dropout` y con `Graduate`.

> ⚠️ Las categóricas codificadas (curso, ocupación, etc.) son **códigos nominales**: su correlación numérica no tiene sentido, por eso se excluyen.

In [ ]:
corr = df[NUM_COLS].corr(method="pearson")
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(15, 12))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
            annot_kws={"size": 7}, linewidths=0.3, cbar_kws={"shrink": 0.8})
plt.title("Matriz de correlación de Pearson (variables numéricas)", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Pares con alta correlación (posible multicolinealidad)
pares = corr.where(~mask).stack().reset_index()
pares.columns = ["var1", "var2", "r"]
pares["abs_r"] = pares["r"].abs()
altos = pares[pares["abs_r"] >= 0.7].sort_values("abs_r", ascending=False)
print(f"Pares con |r| >= 0.70: {len(altos)}")
display(altos.drop(columns="abs_r"))

In [ ]:
# Correlación con el target (one-vs-rest)
y_drop = (df[TARGET] == "Dropout").astype(int)
y_grad = (df[TARGET] == "Graduate").astype(int)
cols_corr = NUM_COLS + BINARY_COLS

corr_t = pd.DataFrame({"dropout": df[cols_corr].corrwith(y_drop),
                       "graduate": df[cols_corr].corrwith(y_grad)}).sort_values("dropout")

corr_t.plot.barh(figsize=(10, 12), width=0.8)
plt.axvline(0, color="black", lw=0.8)
plt.title("Correlación de cada variable con Dropout / Graduate"); plt.tight_layout(); plt.show()

print("Top 8 asociadas a mayor deserción:")
display(corr_t["dropout"].sort_values(ascending=False).head(8).to_frame())

## 1.13 Chequeo de consistencia lógica
Reglas de negocio que deberían cumplirse:
- **Duras** (error seguro): notas fuera de rango (ingreso y previa en [0, 200]; notas semestrales en [0, 20]), valores negativos en conteos.
- **Blandas** (sospechosas, no necesariamente error): más aprobadas que inscritas, nota > 0 sin materias inscritas, etc.

In [ ]:
def reglas_consistencia(d):
    s1, s2 = "curricular_units_1st_sem_", "curricular_units_2nd_sem_"
    duras = {
        "admission_grade fuera de [0, 200]": ~d["admission_grade"].between(0, 200),
        "previous_qualification_grade fuera de [0, 200]": ~d["previous_qualification_grade"].between(0, 200),
        "nota 1er sem fuera de [0, 20]": ~d[s1 + "grade"].between(0, 20),
        "nota 2do sem fuera de [0, 20]": ~d[s2 + "grade"].between(0, 20),
        "edad al matricular < 14": d["age_at_enrollment"] < 14,
        "conteos curriculares negativos": (d[[c for c in d.columns if c.startswith("curricular_units")
                                               and "grade" not in c]] < 0).any(axis=1),
    }
    blandas = {
        "1er sem: aprobadas > inscritas": d[s1 + "approved"] > d[s1 + "enrolled"],
        "2do sem: aprobadas > inscritas": d[s2 + "approved"] > d[s2 + "enrolled"],
        "1er sem: nota > 0 sin materias inscritas": (d[s1 + "grade"] > 0) & (d[s1 + "enrolled"] == 0),
        "2do sem: nota > 0 sin materias inscritas": (d[s2 + "grade"] > 0) & (d[s2 + "enrolled"] == 0),
        "1er sem: inscritas > 0 y nota == 0": (d[s1 + "enrolled"] > 0) & (d[s1 + "grade"] == 0),
        "2do sem: inscritas > 0 y nota == 0": (d[s2 + "enrolled"] > 0) & (d[s2 + "grade"] == 0),
        "sin materias inscritas en ambos semestres": (d[s1 + "enrolled"] == 0) & (d[s2 + "enrolled"] == 0),
    }
    return duras, blandas


duras, blandas = reglas_consistencia(df)
tabla_reglas = pd.DataFrame(
    [{"tipo": "dura", "regla": k, "filas": int(v.sum())} for k, v in duras.items()] +
    [{"tipo": "blanda", "regla": k, "filas": int(v.sum())} for k, v in blandas.items()])
tabla_reglas["pct"] = (tabla_reglas["filas"] / len(df) * 100).round(2)
display(tabla_reglas)

## 1.14 Resumen automático de hallazgos del EDA

In [ ]:
print("=" * 70)
print("RESUMEN DEL EDA")
print("=" * 70)
print(f"• Registros: {len(df):,} | Variables: {len(FEATURES)} (+ target)")
print(f"• Nulos totales: {total_nulos} | Duplicados exactos: {n_dup}")
print(f"• Clases del target: " + ", ".join(f"{k} {v:.1f}%" for k, v in (df[TARGET].value_counts(normalize=True) * 100).items()))
print(f"• Variables numéricas muy sesgadas (|skew|>1): {int((desc['asimetria'].abs() > 1).sum())} de {len(NUM_COLS)}")
print(f"• Variables con outliers IQR: {int((tabla_outliers(df, NUM_COLS)['n_outliers'] > 0).sum())} de {len(NUM_COLS)}")
print(f"• Pares muy correlacionados (|r|>=0.7): {len(altos)}")
print(f"• Violaciones de reglas duras: {int(tabla_reglas.query('tipo == \"dura\"')['filas'].sum())}")
print(f"• Casos sospechosos (reglas blandas): {int(tabla_reglas.query('tipo == \"blanda\"')['filas'].sum())} (suma, una fila puede repetirse)")

### 📝 Conclusiones del EDA
1. **Calidad de los datos:** el dataset llega limpio: **0 nulos, 0 duplicados y 0 violaciones de reglas duras** (notas fuera de rango, conteos negativos). Solo se detectaron **912 estudiantes (≈ 21 %) con casos atípicos** (materias inscritas con nota 0, o sin materias en ambos semestres), que se marcan sin eliminar.
2. **Target desbalanceado:** Graduate 49,9 %, Dropout 32,1 %, Enrolled 17,9 % (razón mayoritaria/minoritaria ≈ 2,8). Obliga a usar partición **estratificada**, `class_weight` y métricas robustas como macro-F1 en lugar de solo accuracy.
3. **Distribuciones:** 8 de 18 variables numéricas están muy sesgadas (|asimetría| > 1): unidades **sin evaluación** (8,2 y 7,3), **créditos reconocidos** (~4,2–4,6), **edad al ingresar** (2,1) y las **notas semestrales** (sesgo negativo, por los estudiantes con nota 0). Las notas de ingreso y previa son aproximadamente simétricas. Los conteos curriculares tienen alta dispersión relativa (CV > 300 %) por la gran cantidad de ceros; el CV de `gdp` e `inflation_rate` no es interpretable porque su media está cerca de 0.
4. **Correlaciones:** la señal más fuerte con la deserción es el rendimiento académico: unidades aprobadas y notas del 2.º semestre (r ≈ −0,57) y del 1.º (r ≈ −0,48), seguidas de matrícula al día (−0,43) y beca (−0,25). Se asocian positivamente la edad al ingresar (0,25), la deuda (0,23) y el género codificado como 1 (0,20). Hay **multicolinealidad** fuerte entre variables del 1.er y 2.º semestre (11 pares con |r| ≥ 0,7; máx. 0,945), lo que favorece modelos regularizados o de árboles.
5. **Outliers:** 15 de 18 variables tienen valores fuera del rango IQR, pero la mayoría son **casos legítimos** (estudiantes adultos, conteos altos de materias). Solo se winsorizaron notas de ingreso/previa y edad.
6. **Variables que ya sugieren riesgo:** matrícula atrasada (87 % de deserción vs. 25 %), deuda (62 % vs. 28 %), sin beca (39 % vs. 12 %) y mayor edad al ingresar. La deserción varía mucho por curso (de ~15 % a ~55 %; el curso 33 tiene solo 12 estudiantes y no es concluyente).

---
# 🧹 PARTE 2 — Limpieza y Preprocesamiento
Siguiendo el flujo de los talleres de limpieza: **auditar → duplicados → nulos → texto/tipos → consistencia → outliers → validar → exportar**.
Todo se hace sobre una **copia** (`df_clean`); `df` / `df_raw` quedan intactos para comparar.

## 2.1 Copia de trabajo y bitácora

In [ ]:
df_clean = df.copy()
bitacora = {"filas_iniciales": len(df_clean), "columnas_iniciales": df_clean.shape[1]}
print(bitacora)

## 2.2 Estandarización del target (texto)

In [ ]:
# Igual que con 'nombre'/'departamento' en el taller: quitar espacios y unificar formato
df_clean[TARGET] = df_clean[TARGET].astype(str).str.strip().str.title()

# Un registro sin target no sirve para aprendizaje supervisado y no se puede imputar
sin_target = df_clean[TARGET].isin(["Nan", "", "None"])
bitacora["filas_sin_target_eliminadas"] = int(sin_target.sum())
df_clean = df_clean.loc[~sin_target].copy()

print("Categorías del target:", sorted(df_clean[TARGET].unique()))
print("Filas sin target eliminadas:", bitacora["filas_sin_target_eliminadas"])

## 2.3 Eliminación de duplicados

In [ ]:
antes = len(df_clean)
df_clean = df_clean.drop_duplicates(keep="first").reset_index(drop=True)
bitacora["duplicados_eliminados"] = antes - len(df_clean)
print(f"Duplicados eliminados: {bitacora['duplicados_eliminados']} | Filas restantes: {len(df_clean):,}")

## 2.4 Tratamiento de valores nulos
Estrategia (la misma lógica de los talleres, automatizada):
1. Columna con **> 40 %** de nulos → se elimina.
2. Numéricas → **mediana** (robusta a sesgo y outliers, mejor que la media aquí).
3. Binarias / categóricas codificadas → **moda**.
4. Alternativa multivariada: `KNNImputer` (comentada abajo).

Este dataset suele venir sin nulos, pero el código queda listo y reutilizable.

In [ ]:
UMBRAL_COLUMNA = 0.40

pct_nulos = df_clean.isna().mean()
cols_eliminar = pct_nulos[pct_nulos > UMBRAL_COLUMNA].index.tolist()
if cols_eliminar:
    df_clean = df_clean.drop(columns=cols_eliminar)
bitacora["columnas_eliminadas_por_nulos"] = cols_eliminar

num_act = [c for c in NUM_COLS if c in df_clean.columns]
cat_act = [c for c in BINARY_COLS + CAT_COLS if c in df_clean.columns]

imputados = {}
for c in df_clean.columns[df_clean.isna().any()]:
    n = int(df_clean[c].isna().sum())
    if c in num_act:
        df_clean[c] = df_clean[c].fillna(df_clean[c].median())
        imputados[c] = (n, "mediana")
    elif c in cat_act:
        df_clean[c] = df_clean[c].fillna(df_clean[c].mode()[0])
        imputados[c] = (n, "moda")

bitacora["nulos_imputados"] = imputados
print("Columnas eliminadas por exceso de nulos:", cols_eliminar or "ninguna")
print("Imputaciones realizadas:", imputados or "ninguna (no había nulos)")
print("Nulos restantes:", int(df_clean.isna().sum().sum()))

# --- Alternativa multivariada (como en el taller 2) ---
# from sklearn.impute import KNNImputer
# df_clean[num_act] = KNNImputer(n_neighbors=5).fit_transform(df_clean[num_act])

## 2.5 Corrección de tipos de dato (y ahorro de memoria)

In [ ]:
mem_antes = df_clean.memory_usage(deep=True).sum() / 1024

for c in BINARY_COLS:
    df_clean[c] = df_clean[c].astype("int8")
for c in CAT_COLS:
    df_clean[c] = df_clean[c].astype("int16")
for c in num_act:
    if (df_clean[c] % 1 == 0).all():
        df_clean[c] = pd.to_numeric(df_clean[c], downcast="integer")   # conteos y edad -> enteros

mem_despues = df_clean.memory_usage(deep=True).sum() / 1024
print(f"Memoria: {mem_antes:,.1f} KB -> {mem_despues:,.1f} KB")
print(df_clean.dtypes.value_counts())

## 2.6 Consistencia lógica
- **Reglas duras** → registros imposibles: se **eliminan**.
- **Reglas blandas** → casos atípicos pero posibles (p. ej. un desertor con 0 materias inscritas): **no se borran**, se **marcan** en `flag_inconsistencia` para poder auditarlos o probar el modelo con y sin ellos.

In [ ]:
duras, blandas = reglas_consistencia(df_clean)

invalidos = pd.concat(duras.values(), axis=1).any(axis=1)
bitacora["filas_invalidas_eliminadas"] = int(invalidos.sum())
df_clean = df_clean.loc[~invalidos].reset_index(drop=True)

_, blandas = reglas_consistencia(df_clean)
df_clean["flag_inconsistencia"] = pd.concat(blandas.values(), axis=1).any(axis=1).astype("int8")
bitacora["filas_marcadas_sospechosas"] = int(df_clean["flag_inconsistencia"].sum())

print(f"Filas inválidas eliminadas: {bitacora['filas_invalidas_eliminadas']}")
print(f"Filas marcadas como sospechosas (flag_inconsistencia=1): {bitacora['filas_marcadas_sospechosas']} "
      f"({df_clean['flag_inconsistencia'].mean() * 100:.1f}%)")
display(df_clean.groupby(TARGET)["flag_inconsistencia"].mean().mul(100).round(2).to_frame("% sospechosas por clase"))

## 2.7 Outliers: IQR + winsorización
**Criterio:** se winsoriza solo donde un valor extremo puede ser ruido o distorsionar escalas (`admission_grade`, `previous_qualification_grade`, `age_at_enrollment`).
Los **conteos de unidades curriculares** y las **notas semestrales** *no* se tocan: ceros y valores altos son comportamiento académico real (p. ej. quien deserta tiene 0 aprobadas).

Se puede desactivar con `APLICAR_WINSORIZACION = False`.

In [ ]:
APLICAR_WINSORIZACION = True
K_IQR = 1.5
COLS_WINSOR = ["previous_qualification_grade", "admission_grade", "age_at_enrollment"]

limites = {}
for c in COLS_WINSOR:
    q1, q3 = df_clean[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    li, ls = q1 - K_IQR * iqr, q3 + K_IQR * iqr
    n_out = int(((df_clean[c] < li) | (df_clean[c] > ls)).sum())
    limites[c] = {"lim_inf": li, "lim_sup": ls, "n_ajustados": n_out}

display(pd.DataFrame(limites).T)

df_antes = df_clean[COLS_WINSOR].copy()
if APLICAR_WINSORIZACION:
    for c, v in limites.items():
        df_clean[c] = df_clean[c].clip(lower=v["lim_inf"], upper=v["lim_sup"])
    df_clean["age_at_enrollment"] = df_clean["age_at_enrollment"].round().astype("int16")
bitacora["winsorizacion"] = {c: v["n_ajustados"] for c, v in limites.items()} if APLICAR_WINSORIZACION else "desactivada"

# Comparación visual antes vs después
fig, axes = plt.subplots(2, 3, figsize=(17, 7))
for j, c in enumerate(COLS_WINSOR):
    sns.boxplot(x=df_antes[c], ax=axes[0, j], color="lightcoral"); axes[0, j].set_title(f"ANTES · {c}", fontsize=10)
    sns.boxplot(x=df_clean[c], ax=axes[1, j], color="mediumseagreen"); axes[1, j].set_title(f"DESPUÉS · {c}", fontsize=10)
    axes[0, j].set_xlabel(""); axes[1, j].set_xlabel("")
plt.tight_layout(); plt.show()

## 2.8 Codificación del target

In [ ]:
MAPA_TARGET = {"Dropout": 0, "Enrolled": 1, "Graduate": 2}
df_clean["target_cod"] = df_clean[TARGET].map(MAPA_TARGET)
assert df_clean["target_cod"].notna().all(), "Hay categorías de target no reconocidas"
df_clean["target_cod"] = df_clean["target_cod"].astype("int8")

# Versión binaria útil para un primer modelo (¿abandona o no?)
df_clean["abandono"] = (df_clean[TARGET] == "Dropout").astype("int8")

display(df_clean[[TARGET, "target_cod", "abandono"]].drop_duplicates().sort_values("target_cod"))

## 2.9 Validación de calidad (aserciones)

In [ ]:
assert df_clean.isna().sum().sum() == 0, "Persisten nulos"
assert df_clean.drop(columns=["flag_inconsistencia"]).duplicated().sum() == 0, "Persisten duplicados"
assert df_clean["admission_grade"].between(0, 200).all(), "admission_grade fuera de rango"
assert df_clean["previous_qualification_grade"].between(0, 200).all(), "previous_qualification_grade fuera de rango"
assert df_clean["curricular_units_1st_sem_grade"].between(0, 20).all(), "nota 1er sem fuera de rango"
assert df_clean["curricular_units_2nd_sem_grade"].between(0, 20).all(), "nota 2do sem fuera de rango"
assert (df_clean["age_at_enrollment"] >= 14).all(), "edad inválida"
assert set(df_clean[TARGET].unique()) <= set(MAPA_TARGET), "categorías de target inesperadas"
print("✅ Todas las validaciones de calidad pasaron.")

## 2.10 Comparación antes vs. después

In [ ]:
comparacion = pd.DataFrame({
    "antes (df)": [len(df), df.shape[1], int(df.isna().sum().sum()), int(df.duplicated().sum())],
    "después (df_clean)": [len(df_clean), df_clean.shape[1], int(df_clean.isna().sum().sum()), int(df_clean.duplicated().sum())],
}, index=["filas", "columnas", "nulos", "duplicados"])
display(comparacion)

print("\nBitácora de limpieza:")
print(json.dumps(bitacora, indent=2, ensure_ascii=False, default=str))

display(df_clean.head())

In [ ]:
# Distribución del target antes vs después (la limpieza no debe sesgar las clases)
cmp = pd.concat([df[TARGET].value_counts(normalize=True).mul(100).rename("antes_%"),
                 df_clean[TARGET].value_counts(normalize=True).mul(100).rename("despues_%")], axis=1)
display(cmp.round(2))

## 2.11 Exportación del dataset limpio

In [ ]:
ARCHIVO_CSV = "estudiantes_limpio.csv"
df_clean.to_csv(ARCHIVO_CSV, index=False)
print(f"CSV guardado: {ARCHIVO_CSV} ({len(df_clean):,} filas)")

try:
    df_clean.to_parquet("estudiantes_limpio.parquet", index=False)
    print("Parquet guardado: estudiantes_limpio.parquet")
except Exception as e:
    print("No se pudo guardar Parquet (opcional):", e)

# Metadatos de columnas para reutilizar en el modelado y en Streamlit
meta = {"target": TARGET, "target_cod": "target_cod", "mapa_target": MAPA_TARGET,
        "binarias": BINARY_COLS, "categoricas": CAT_COLS, "numericas": num_act}
with open("metadata_columnas.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print("Metadatos guardados: metadata_columnas.json")

# En Colab: descarga automática a tu computador
try:
    from google.colab import files
    for f in [ARCHIVO_CSV, "metadata_columnas.json"]:
        files.download(f)
except ImportError:
    pass

---
## ✅ Resumen de decisiones de limpieza (para el póster y la sustentación)

| Aspecto | Decisión | Justificación |
|---|---|---|
| Nombres | `snake_case`, corrección `Nacionality → nationality` | Evitar errores por espacios/tab ocultos |
| Duplicados | Eliminar exactos | Evitan sobre-representar casos |
| Nulos | Mediana (num) / moda (cat); columna > 40 % se elimina | Robusto a sesgo; el dataset original casi no tiene |
| Tipos | `int8`/`int16`/enteros | Menor memoria, tipos coherentes |
| Consistencia | Duras → eliminar; blandas → `flag_inconsistencia` | No perder desertores reales con 0 materias |
| Outliers | Winsorización IQR solo en notas de ingreso/previa y edad | Conteos y notas semestrales son señal real |
| Target | `target_cod` (0/1/2) y `abandono` (binario) | Listo para clasificación multiclase o binaria |

### ➡️ Siguiente paso
Continúa con la **Parte 3: Preparación para modelado** (más abajo).

---
# 🧪 PARTE 3 — Preparación del dataset para modelado
Aquí dejamos todo listo para entrenar: partición **estratificada** por clase, validación cruzada y *pipelines* de preprocesamiento que aprenden **solo del entrenamiento** (sin fuga de datos).

### Modelos previstos y por qué (justificación completa en la sección 4.0)
| Modelo | Preprocesamiento | Qué aporta al análisis |
|---|---|---|
| **Regresión Logística** (regularizada) | escalado + one-hot | Referencia; coeficientes interpretables |
| **Random Forest** | sin escalado, categóricas ordinales | Robusto y estable; importancia de variables |
| **Gradient Boosting** (`HistGradientBoosting`) | sin escalado, categóricas ordinales | Mejor rendimiento esperado; candidato para Streamlit |

**Métricas comunes:** accuracy, macro-F1, balanced accuracy, F1 por clase y matriz de confusión, con validación cruzada de 5 folds (la desviación entre folds indica qué tan estable es la conclusión).

## 3.1 Dataset de modelado: variables (X) y objetivo (y)

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

# ---- Parámetros editables ----
OBJETIVO = "target_cod"              # "target_cod" = 3 clases (0 Dropout, 1 Enrolled, 2 Graduate) | "abandono" = binario
USAR_FLAG_INCONSISTENCIA = False     # pruébalo en True para comparar el efecto en el modelo
TEST_SIZE = 0.20
N_FOLDS = 5

df_model = df_clean.copy()
# Restauramos los valores SIN winsorizar: el recorte IQR se aprenderá después solo con el train (evita fuga de datos)
df_model[COLS_WINSOR] = df_antes[COLS_WINSOR]

excluir = [TARGET, "target_cod", "abandono"] + ([] if USAR_FLAG_INCONSISTENCIA else ["flag_inconsistencia"])
features = [c for c in df_model.columns if c not in excluir]

bin_cols = BINARY_COLS + (["flag_inconsistencia"] if USAR_FLAG_INCONSISTENCIA else [])
cat_cols = list(CAT_COLS)
num_cols = [c for c in num_act]
assert set(bin_cols + cat_cols + num_cols) == set(features), "Hay columnas sin clasificar"

X = df_model[features]
y = df_model[OBJETIVO]
ETIQUETAS = {0: "Dropout", 1: "Enrolled", 2: "Graduate"} if OBJETIVO == "target_cod" else {0: "No abandona", 1: "Abandona"}

print(f"X: {X.shape} | y: {y.shape} | objetivo = '{OBJETIVO}'")
print(f"Numéricas: {len(num_cols)} | Binarias: {len(bin_cols)} | Categóricas codificadas: {len(cat_cols)}")

## 3.2 Partición estratificada (porcentajes por categoría)
`stratify=y` hace que **cada subconjunto conserve el mismo porcentaje de cada clase** que el dataset completo. Es clave aquí porque `Enrolled` es solo ~18 %.
Se usa **80 % entrenamiento / 20 % prueba**; el conjunto de prueba no se toca hasta la evaluación final.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)

tabla_split = pd.DataFrame({
    "total_%": y.value_counts(normalize=True).sort_index() * 100,
    "train_%": y_train.value_counts(normalize=True).sort_index() * 100,
    "test_%": y_test.value_counts(normalize=True).sort_index() * 100,
    "n_total": y.value_counts().sort_index(),
    "n_train": y_train.value_counts().sort_index(),
    "n_test": y_test.value_counts().sort_index(),
})
tabla_split.index = [ETIQUETAS[i] for i in tabla_split.index]
display(tabla_split.round(2))
print(f"Train: {len(X_train):,} filas ({len(X_train) / len(X) * 100:.0f}%) | Test: {len(X_test):,} filas ({len(X_test) / len(X) * 100:.0f}%)")

tabla_split[["total_%", "train_%", "test_%"]].plot.bar(rot=0, figsize=(9, 4))
plt.title("Proporción de cada clase por conjunto (deben ser casi idénticas)"); plt.ylabel("%"); plt.show()

## 3.3 Validación cruzada estratificada
Dentro del entrenamiento se usan **5 folds estratificados**: cada modelo se evalúa 5 veces y se reporta media ± desviación. Así la conclusión no depende de una sola partición.

In [ ]:
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

filas = []
for k, (idx_tr, idx_va) in enumerate(cv.split(X_train, y_train), start=1):
    prop = y_train.iloc[idx_va].value_counts(normalize=True).sort_index() * 100
    filas.append({"fold": k, "n_validacion": len(idx_va), **{ETIQUETAS[i]: round(v, 1) for i, v in prop.items()}})
display(pd.DataFrame(filas).set_index("fold"))

## 3.4 Desbalance de clases: pesos
En lugar de inventar datos (SMOTE), se usará `class_weight="balanced"`: penaliza más los errores en las clases minoritarias. Estos son los pesos que se aplicarán.

In [ ]:
clases = np.sort(y_train.unique())
pesos = compute_class_weight("balanced", classes=clases, y=y_train)
display(pd.DataFrame({"clase": [ETIQUETAS[c] for c in clases], "peso": pesos.round(3)}).set_index("clase"))

## 3.5 Pipelines de preprocesamiento (sin fuga de datos)
Todo lo que se "aprende" de los datos (límites IQR, media/desviación del escalado, categorías del one-hot) se ajusta **solo con el entrenamiento** y luego se aplica al test.

- `WinsorizadorIQR`: recorta `previous_qualification_grade`, `admission_grade` y `age_at_enrollment` con límites calculados en el train.
- **Para modelos lineales** (Logística): numéricas escaladas + categóricas en one-hot.
- **Para modelos de árboles** (RF, Boosting): sin escalado + categóricas ordinales.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder


class WinsorizadorIQR(BaseEstimator, TransformerMixin):
    """Recorta valores fuera de [Q1 - k*IQR, Q3 + k*IQR]; los límites se aprenden en fit (solo train)."""
    def __init__(self, columnas, k=1.5):
        self.columnas = columnas
        self.k = k

    def fit(self, X, y=None):
        self.limites_ = {}
        for c in self.columnas:
            q1, q3 = X[c].quantile([0.25, 0.75])
            iqr = q3 - q1
            self.limites_[c] = (q1 - self.k * iqr, q3 + self.k * iqr)
        return self

    def transform(self, X):
        X = X.copy()
        for c, (li, ls) in self.limites_.items():
            X[c] = X[c].clip(li, ls)
        return X


def crear_preprocesador(tipo, excluir=()):
    """tipo = 'lineal' (escalado + one-hot) o 'arboles' (sin escalado + ordinal).
    excluir: columnas que NO deben entrar al modelo (se descartan)."""
    num = [c for c in num_cols if c not in excluir]
    cat = [c for c in cat_cols if c not in excluir]
    binc = [c for c in bin_cols if c not in excluir]
    if tipo == "lineal":
        columnas = ColumnTransformer([
            ("num", StandardScaler(), num),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
            ("bin", "passthrough", binc),
        ])
    else:
        columnas = ColumnTransformer([
            ("num", "passthrough", num),
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat),
            ("bin", "passthrough", binc),
        ])
    return Pipeline([("winsor", WinsorizadorIQR(COLS_WINSOR, k=K_IQR)), ("columnas", columnas)])


prep_lineal = crear_preprocesador("lineal")
prep_arboles = crear_preprocesador("arboles")
print("Preprocesadores creados: prep_lineal, prep_arboles")

## 3.6 Verificación: el preprocesamiento funciona y no hay fuga

In [ ]:
Xtr_lin = prep_lineal.fit_transform(X_train)     # se ajusta SOLO con el train
Xte_lin = prep_lineal.transform(X_test)           # el test solo se transforma
Xtr_arb = prep_arboles.fit_transform(X_train)
Xte_arb = prep_arboles.transform(X_test)

print(f"Lineal  -> train {Xtr_lin.shape} | test {Xte_lin.shape}  (one-hot expande {len(cat_cols)} categóricas)")
print(f"Árboles -> train {Xtr_arb.shape} | test {Xte_arb.shape}")
assert not np.isnan(Xtr_lin).any() and not np.isnan(Xte_lin).any(), "Hay NaN tras transformar"
assert not np.isnan(Xtr_arb).any() and not np.isnan(Xte_arb).any(), "Hay NaN tras transformar"
assert Xtr_lin.shape[1] == Xte_lin.shape[1] and Xtr_arb.shape[1] == Xte_arb.shape[1]

# Las numéricas escaladas deben tener media ~0 y std ~1 en el TRAIN (y aproximada en el test)
n = len(num_cols)
print(f"\nNuméricas escaladas | train: media={Xtr_lin[:, :n].mean():.3f}, std={Xtr_lin[:, :n].std():.3f} "
      f"| test: media={Xte_lin[:, :n].mean():.3f}, std={Xte_lin[:, :n].std():.3f}")

# Límites de winsorización aprendidos solo con el train
lim = pd.DataFrame(prep_lineal.named_steps["winsor"].limites_, index=["lim_inf", "lim_sup"]).T
display(lim.round(2))

## 3.7 Guardar los conjuntos para el modelado y el Streamlit

In [ ]:
train_df = X_train.assign(**{OBJETIVO: y_train})
test_df = X_test.assign(**{OBJETIVO: y_test})
train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

meta_modelado = {"objetivo": OBJETIVO, "etiquetas": {int(k): v for k, v in ETIQUETAS.items()},
                 "numericas": num_cols, "binarias": bin_cols, "categoricas": cat_cols,
                 "cols_winsor": COLS_WINSOR, "k_iqr": K_IQR, "test_size": TEST_SIZE,
                 "n_folds": N_FOLDS, "seed": SEED}
with open("metadata_modelado.json", "w", encoding="utf-8") as f:
    json.dump(meta_modelado, f, indent=2, ensure_ascii=False)
print(f"Guardados: train.csv ({len(train_df):,} filas), test.csv ({len(test_df):,} filas), metadata_modelado.json")

try:
    from google.colab import files
    for f in ["train.csv", "test.csv", "metadata_modelado.json"]:
        files.download(f)
except ImportError:
    pass

### ✅ Resumen de la preparación
| Decisión | Valor | Justificación |
|---|---|---|
| Partición | 80 % train / 20 % test, **estratificada** | Mantiene el % de cada clase (incluida `Enrolled`) |
| Validación | StratifiedKFold de 5 folds | Conclusiones estables (media ± desviación) |
| Desbalance | `class_weight="balanced"` | Sin inventar datos sintéticos |
| Fuga de datos | Winsorización, escalado y one-hot dentro del `Pipeline` | El test nunca influye en el preprocesamiento |
| Modelos | Logística, Random Forest, Gradient Boosting | Interpretables, estables y comparables |

### ➡️ Siguiente paso
Continúa con la **Parte 4: modelado y evaluación**.

---
# 🤖 PARTE 4 — Modelado clásico y evaluación
Se entrenan y comparan **3 modelos** con el mismo protocolo, para que la comparación sea justa:

1. **Regresión Logística** regularizada (referencia, interpretable)
2. **Random Forest** (robusto y estable)
3. **Gradient Boosting** (`HistGradientBoosting`, mejor rendimiento esperado)

**Protocolo:**
`CV base (5 folds)` → `ajuste de hiperparámetros con GridSearchCV (métrica macro-F1)` → `evaluación final única sobre el test` → `interpretación`.

> **Métrica principal: macro-F1.** Promedia el F1 de las 3 clases con el mismo peso, así que no se deja engañar por la clase mayoritaria (`Graduate`, ~50 %). Se acompaña de balanced accuracy y accuracy.

## 4.0 ¿Por qué estos tres modelos?
La elección no es arbitraria: sale de lo que mostró el EDA y de comparar **tres familias de algoritmos distintas** (lineal, *bagging* y *boosting*). Si familias tan diferentes llegan a un rendimiento parecido, la conclusión es **estable** y no depende de un algoritmo particular.

| | **Regresión Logística** (regularizada) | **Random Forest** | **Gradient Boosting** |
|---|---|---|---|
| **Familia** | Lineal / probabilística | Ensamble *bagging*: muchos árboles independientes que votan | Ensamble *boosting*: árboles en secuencia, cada uno corrige los errores del anterior |
| **Caso de uso típico** | Clasificación y *scoring* de riesgo donde se necesita explicar la decisión (crédito, diagnóstico) | Clasificación sobre tablas con variables de distinto tipo (fuga de clientes, fraude) | Predicción sobre datos tabulares cuando se busca la máxima precisión (riesgo, deserción, *churn*) |
| **Fortaleza** | Interpretable (cada coeficiente indica el efecto de la variable), rápida y poco propensa al sobreajuste | Capta relaciones no lineales e interacciones, no necesita escalar y es robusta a outliers | Suele ser el más preciso en datos tabulares; admite pesos por clase |
| **Limitación** | Supone fronteras lineales; exige escalado y one-hot; no capta interacciones por sí sola | Menos interpretable; tiende a memorizar el entrenamiento (sobreajuste) | Más hiperparámetros que ajustar; riesgo de sobreajuste; menos interpretable |
| **Por qué encaja aquí** | El EDA mostró **multicolinealidad** fuerte (11 pares con \|r\| ≥ 0,7; máx. 0,945 entre semestres): la regularización L2 la controla. Sirve de **referencia** y explica *qué* variables elevan la deserción | **15 de 18** numéricas tienen outliers y 8 están muy sesgadas: los árboles no se ven afectados por la escala ni por valores extremos. Maneja la mezcla de numéricas, binarias y categóricas codificadas | Es el candidato natural a mejor rendimiento y a **modelo final**; 4.424 registros son suficientes para entrenarlo sin problema |
| **Preprocesamiento** | Escalado + one-hot | Ninguno (categóricas ordinales) | Ninguno (categóricas ordinales) |

**Modelos considerados y descartados**
- **Regresión lineal:** es para predecir valores continuos; aquí el objetivo es una clase.
- **Árbol de decisión simple:** es inestable y sobreajusta; el Random Forest lo mejora y lo incluye.
- **KNN:** sufre con muchas variables mezcladas y correlacionadas (243 columnas tras el one-hot) y es lento al predecir.
- **SVM (kernel RBF):** posible, pero más costoso, exige ajustar `C` y `gamma`, y es poco interpretable; no aporta una familia nueva frente a los tres elegidos.
- **Redes neuronales / deep learning:** quedan fuera del alcance de este ACA (clásico).

**Criterio de comparación:** los tres se evalúan con el mismo protocolo (mismas particiones, mismas métricas y macro-F1 como métrica principal), de modo que cualquier diferencia se deba al modelo y no al procedimiento.

## 4.1 Definición de los modelos

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import cross_validate, GridSearchCV
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                             classification_report, ConfusionMatrixDisplay)

SCORING = {"accuracy": "accuracy", "balanced_accuracy": "balanced_accuracy", "f1_macro": "f1_macro"}
ES_MULTICLASE = y.nunique() > 2
CLASES = sorted(y.unique())
NOMBRES_CLASES = [ETIQUETAS[c] for c in CLASES]

modelos_base = {
    "Regresión Logística": Pipeline([
        ("prep", crear_preprocesador("lineal")),
        ("modelo", LogisticRegression(class_weight="balanced", max_iter=3000, random_state=SEED))]),
    "Random Forest": Pipeline([
        ("prep", crear_preprocesador("arboles")),
        ("modelo", RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                          random_state=SEED, n_jobs=-1))]),
    "Gradient Boosting": Pipeline([
        ("prep", crear_preprocesador("arboles")),
        ("modelo", HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED))]),
}
print("Modelos definidos:", list(modelos_base))

## 4.2 Validación cruzada base (hiperparámetros por defecto)
Cada modelo se evalúa en los mismos 5 folds estratificados. Se reporta **media ± desviación**: una desviación pequeña indica un resultado **estable**.
La *brecha train–val* en macro-F1 mide el sobreajuste (mucho más alto en train que en validación = memoriza).

In [ ]:
def resumen_cv(resultados):
    filas = {}
    for nombre, r in resultados.items():
        filas[nombre] = {
            "accuracy": r["test_accuracy"].mean(), "accuracy_std": r["test_accuracy"].std(),
            "bal_accuracy": r["test_balanced_accuracy"].mean(), "bal_accuracy_std": r["test_balanced_accuracy"].std(),
            "f1_macro": r["test_f1_macro"].mean(), "f1_macro_std": r["test_f1_macro"].std(),
            "f1_macro_train": r["train_f1_macro"].mean(),
        }
    d = pd.DataFrame(filas).T
    d["brecha_train_val"] = d["f1_macro_train"] - d["f1_macro"]
    return d


resultados_base = {n: cross_validate(m, X_train, y_train, cv=cv, scoring=SCORING,
                                     return_train_score=True, n_jobs=-1)
                   for n, m in modelos_base.items()}
tabla_cv_base = resumen_cv(resultados_base)
display(tabla_cv_base.round(3))

## 4.3 Ajuste de hiperparámetros (GridSearchCV)
Rejilla pequeña y razonable para cada modelo, optimizando **macro-F1** con los mismos 5 folds. Las rejillas son modestas a propósito: más combinaciones mejoran poco y aumentan el riesgo de sobreajustar la validación.

In [ ]:
grids = {
    "Regresión Logística": {"modelo__C": [0.01, 0.03, 0.1, 0.3, 1, 3]},
    "Random Forest": {"modelo__max_depth": [8, 12, None],
                      "modelo__min_samples_leaf": [1, 3, 5],
                      "modelo__max_features": ["sqrt", 0.5]},
    "Gradient Boosting": {"modelo__learning_rate": [0.03, 0.1],
                          "modelo__max_depth": [3, 6],
                          "modelo__min_samples_leaf": [20, 40],
                          "modelo__max_iter": [150, 300]},
}

mejores = {}
for nombre, pipe in modelos_base.items():
    gs = GridSearchCV(pipe, grids[nombre], scoring="f1_macro", cv=cv, n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    mejores[nombre] = gs
    print(f"{nombre:<22} mejor macro-F1 (CV) = {gs.best_score_:.3f} | {gs.best_params_}")

## 4.4 CV de los modelos ajustados vs. base

In [ ]:
resultados_aj = {n: cross_validate(gs.best_estimator_, X_train, y_train, cv=cv, scoring=SCORING,
                                   return_train_score=True, n_jobs=-1)
                 for n, gs in mejores.items()}
tabla_cv_aj = resumen_cv(resultados_aj)
display(tabla_cv_aj.round(3))

mejora = pd.DataFrame({"f1_macro base": tabla_cv_base["f1_macro"], "f1_macro ajustado": tabla_cv_aj["f1_macro"]})
mejora["mejora"] = mejora["f1_macro ajustado"] - mejora["f1_macro base"]
display(mejora.round(3))

# Estabilidad: distribución del macro-F1 en cada fold
folds = pd.DataFrame({n: r["test_f1_macro"] for n, r in resultados_aj.items()})
plt.figure(figsize=(8, 4))
sns.boxplot(data=folds, color="lightsteelblue")
sns.stripplot(data=folds, color="black", size=6)
plt.title("macro-F1 por fold (modelos ajustados): menor dispersión = más estable"); plt.ylabel("macro-F1")
plt.show()

## 4.5 Evaluación final sobre el TEST
Ahora (y solo ahora) se usa el 20 % reservado. Es la estimación más honesta del desempeño en estudiantes nuevos.
Se agrega **ROC-AUC** (un-vs-resto, promedio macro), que mide qué tan bien separa cada modelo las clases sin depender de un umbral.

In [ ]:
def evaluar(modelo, Xt, yt):
    pred = modelo.predict(Xt)
    proba = modelo.predict_proba(Xt)
    auc = roc_auc_score(yt, proba, multi_class="ovr", average="macro") if ES_MULTICLASE else roc_auc_score(yt, proba[:, 1])
    return pred, {"accuracy": accuracy_score(yt, pred),
                  "balanced_accuracy": balanced_accuracy_score(yt, pred),
                  "f1_macro": f1_score(yt, pred, average="macro"),
                  "f1_weighted": f1_score(yt, pred, average="weighted"),
                  "roc_auc_ovr": auc}


predicciones, metricas_test = {}, {}
for nombre, gs in mejores.items():
    predicciones[nombre], metricas_test[nombre] = evaluar(gs.best_estimator_, X_test, y_test)

tabla_test = pd.DataFrame(metricas_test).T.sort_values("f1_macro", ascending=False)
display(tabla_test.round(3))

tabla_test[["accuracy", "balanced_accuracy", "f1_macro"]].plot.bar(rot=0, figsize=(10, 4.5), ylim=(0, 1))
plt.title("Métricas en el conjunto de prueba"); plt.ylabel("valor"); plt.legend(loc="lower right")
plt.show()

## 4.6 Desempeño por clase y matrices de confusión
La matriz está **normalizada por fila** (cada fila suma 100 %): la diagonal es el *recall* de cada clase, es decir, qué porcentaje de los estudiantes reales de esa clase detecta el modelo.

In [ ]:
# F1 por clase
f1_clase = pd.DataFrame({n: f1_score(y_test, p, average=None, labels=CLASES) for n, p in predicciones.items()},
                        index=NOMBRES_CLASES)
display(f1_clase.round(3))

# Reporte completo (precision / recall / f1) por modelo
for n, p in predicciones.items():
    rep = pd.DataFrame(classification_report(y_test, p, target_names=NOMBRES_CLASES, output_dict=True)).T
    print(f"\n=== {n} ===")
    display(rep.round(3))

# Matrices de confusión
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, (n, p) in zip(axes, predicciones.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, p, display_labels=NOMBRES_CLASES, normalize="true",
                                            values_format=".0%", cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(n)
plt.tight_layout(); plt.show()

## 4.7 Sobreajuste y consistencia CV ↔ Test

In [ ]:
consistencia = pd.DataFrame({
    "f1_macro train (CV)": tabla_cv_aj["f1_macro_train"],
    "f1_macro validación (CV)": tabla_cv_aj["f1_macro"],
    "f1_macro test": tabla_test["f1_macro"],
}).loc[list(mejores)]
consistencia["brecha train-val"] = consistencia["f1_macro train (CV)"] - consistencia["f1_macro validación (CV)"]
consistencia["diferencia val-test"] = consistencia["f1_macro validación (CV)"] - consistencia["f1_macro test"]
display(consistencia.round(3))
print("Lectura: brecha train-val alta = sobreajuste; diferencia val-test cercana a 0 = el resultado es confiable.")

## 4.8 Interpretación: ¿qué variables explican la deserción?
- **Regresión Logística:** coeficientes estandarizados hacia la clase `Dropout` (positivo = aumenta la probabilidad de desertar).
- **Random Forest y Gradient Boosting:** *importancia por permutación* en el test (cuánto cae el macro-F1 al desordenar una variable). Funciona igual en cualquier modelo y evita los sesgos de la importancia por impuridad.

In [ ]:
# --- Coeficientes de la Logística hacia la clase 0 (Dropout) ---
lr = mejores["Regresión Logística"].best_estimator_
nombres = [n.split("__", 1)[1] for n in lr.named_steps["prep"].named_steps["columnas"].get_feature_names_out()]
fila = list(lr.named_steps["modelo"].classes_).index(CLASES[0] if ES_MULTICLASE else 1)
coef = pd.Series(lr.named_steps["modelo"].coef_[fila], index=nombres).sort_values()
top = pd.concat([coef.head(10), coef.tail(10)])

plt.figure(figsize=(9, 7))
top.plot.barh(color=np.where(top > 0, "tomato", "steelblue"))
plt.axvline(0, color="black", lw=0.8)
plt.title("Regresión Logística: mayor efecto sobre Dropout (rojo = más deserción, azul = menos)")
plt.tight_layout(); plt.show()

In [ ]:
# --- Importancia por permutación (RF y Gradient Boosting) ---
imp = {}
for n in ["Random Forest", "Gradient Boosting"]:
    r = permutation_importance(mejores[n].best_estimator_, X_test, y_test, scoring="f1_macro",
                               n_repeats=8, random_state=SEED, n_jobs=-1)
    imp[n] = pd.Series(r.importances_mean, index=X_test.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (n, s) in zip(axes, imp.items()):
    s.head(12).sort_values().plot.barh(ax=ax, color="darkorange")
    ax.set_title(f"{n}: caída del macro-F1 al permutar"); ax.set_xlabel("importancia")
plt.tight_layout(); plt.show()

top_comun = pd.DataFrame({n: s.rank(ascending=False) for n, s in imp.items()}).mean(axis=1).sort_values().head(8)
print("Variables más importantes en común (ranking promedio RF + Boosting):")
display(top_comun.to_frame("ranking_promedio").round(1))

## 4.9 Experimento adicional: ¿se puede avisar a tiempo? (alerta temprana)
Las variables del **2.º semestre** son muy informativas, pero llegan tarde: cuando se conocen, muchos estudiantes ya desertaron.
Aquí se reentrena el mejor Gradient Boosting **sin las variables del 2.º semestre** para ver cuánto se pierde si solo se usa información disponible al terminar el 1.º semestre.

In [ ]:
cols_2do = [c for c in X.columns if "2nd_sem" in c]
params = {k.replace("modelo__", ""): v for k, v in mejores["Gradient Boosting"].best_params_.items()}

gb_temprano = Pipeline([
    ("prep", crear_preprocesador("arboles", excluir=cols_2do)),
    ("modelo", HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED, **params))])
gb_temprano.fit(X_train, y_train)
_, m_temprano = evaluar(gb_temprano, X_test, y_test)

comp = pd.DataFrame({"Gradient Boosting (todas las variables)": metricas_test["Gradient Boosting"],
                     "Gradient Boosting (solo hasta 1.º semestre)": m_temprano}).T
comp["f1_macro: pérdida"] = comp["f1_macro"].iloc[0] - comp["f1_macro"]
display(comp.round(3))
print(f"Variables del 2.º semestre excluidas: {len(cols_2do)}")

## 4.10 Ranking final automático

In [ ]:
orden = tabla_test.sort_values("f1_macro", ascending=False)
mejor = orden.index[0]
print("=" * 70)
print("RESUMEN DE LA EVALUACIÓN (test)")
print("=" * 70)
for i, (n, r) in enumerate(orden.iterrows(), start=1):
    print(f"{i}. {n:<22} macro-F1={r['f1_macro']:.3f} | bal.acc={r['balanced_accuracy']:.3f} | "
          f"acc={r['accuracy']:.3f} | AUC={r['roc_auc_ovr']:.3f} | estabilidad CV (std F1)={tabla_cv_aj.loc[n, 'f1_macro_std']:.3f}")
peor_clase = f1_clase[mejor].idxmin()
print(f"\n→ Mejor modelo por macro-F1: {mejor}")
print(f"→ Clase más difícil para el mejor modelo: {peor_clase} (F1 = {f1_clase.loc[peor_clase, mejor]:.3f})")
print(f"→ Diferencia entre el mejor y el peor modelo: {orden['f1_macro'].iloc[0] - orden['f1_macro'].iloc[-1]:.3f} de macro-F1")

---
# 🏁 PARTE 5 — Conclusiones y borrador de discusión

> Las cifras citadas corresponden a la ejecución con `random_state=42` sobre `data.csv`. Con otras versiones de las librerías pueden variar levemente (±0,01); las conclusiones cualitativas se mantienen.

## 5.1 Resultados principales (conjunto de prueba, 885 estudiantes)

| Modelo | macro-F1 | Balanced acc. | Accuracy | ROC-AUC | macro-F1 en CV (media ± std) | Brecha train–val |
|---|---|---|---|---|---|---|
| **Gradient Boosting** | **0,704** | **0,714** | 0,744 | 0,886 | **0,728 ± 0,011** | 0,19 |
| Regresión Logística | 0,697 | 0,709 | 0,734 | 0,884 | 0,716 ± 0,019 | **0,04** |
| Random Forest | 0,695 | 0,692 | **0,751** | 0,885 | 0,717 ± 0,019 | 0,25 |

**F1 por clase (mejor modelo, Gradient Boosting):** Graduate 0,84 · Dropout 0,75 · Enrolled 0,53.

## 5.2 Conclusiones

1. **Los tres modelos rinden prácticamente igual.** La diferencia entre el mejor y el peor es de ~0,01 de macro-F1, menor que la desviación entre folds (0,01–0,02). No hay evidencia para afirmar que uno supere claramente a otro: el límite lo pone la información disponible en los datos, no el algoritmo. Esa coincidencia entre tres familias distintas (lineal, bagging y boosting) es lo que hace **estable** la conclusión.
2. **El resultado es confiable, no fruto del azar ni de memorizar.** El macro-F1 en validación cruzada (0,72–0,73) y en el test (0,70) son consistentes (diferencia ≈ 0,02, la pequeña caída esperada tras ajustar hiperparámetros). El ROC-AUC de ~0,885 indica buena capacidad de separación de las clases.
3. **Sobreajuste:** Random Forest (train 0,96 vs. validación 0,72) y Gradient Boosting (0,91 vs. 0,73) memorizan parte del entrenamiento; la Regresión Logística casi no (brecha 0,04). Aun así, su rendimiento en validación y test no empeora, por lo que el sobreajuste es tolerable, aunque se podría reducir con más regularización.
4. **Modelo recomendado: Gradient Boosting.** Tiene el mejor macro-F1 y balanced accuracy y el resultado más estable entre folds (std 0,011). Como alternativa, la **Regresión Logística** queda casi a la par (0,697), es la menos sobreajustada y la más fácil de explicar, útil si se prioriza la interpretabilidad.
5. **`Graduate` se predice bien, `Dropout` razonablemente y `Enrolled` es la clase difícil** (F1 ≈ 0,48–0,53). Es la clase minoritaria (18 %) y, por naturaleza, intermedia: un estudiante matriculado puede terminar graduándose o desertando, así que sus características se solapan con las otras dos. La Logística y el Boosting detectan ~65 % de los `Enrolled` pero con precisión baja (~0,43), mientras que el Random Forest acierta más en `Graduate` pero detecta solo ~50 % de los `Enrolled`.
6. **Costo de errores en `Dropout`:** con precisión ≈ 0,83 y recall ≈ 0,69, cuando el modelo anuncia una deserción acierta ~8 de cada 10 veces, pero deja pasar ~3 de cada 10 desertores reales. Para un sistema de apoyo estudiantil conviene bajar el umbral de decisión y aceptar más falsas alarmas a cambio de detectar más casos.
7. **Qué explica la deserción** (coincide entre los coeficientes de la Logística y la importancia por permutación de RF y Boosting):
   - **Rendimiento académico del 2.º y 1.er semestre**: unidades aprobadas y notas son lo más determinante. Con 0 unidades aprobadas en el 2.º semestre, el 84 % deserta; con 4 o más, ≤ 11 %.
   - **Matrícula al día (`tuition_fees_up_to_date`)**: con pagos atrasados deserta el 87 % frente al 25 % de quienes están al día. Estar endeudado (`debtor`) también eleva el riesgo (62 % vs. 28 %).
   - **Beca**: los becarios desertan 12 % frente a 39 % de los no becarios.
   - **Edad al ingresar**: sube el riesgo con la edad (21 % en ≤ 19 años; ~54–60 % en mayores de 25).
   - **Programa académico**: la deserción va de ~15 % (curso 9500) a ~55 % (curso 9130).
8. **Alerta temprana viable:** reentrenando **sin las variables del 2.º semestre** el macro-F1 baja de 0,704 a 0,668 y el AUC de 0,886 a 0,860. Se pierde poco, así que es posible identificar estudiantes en riesgo al terminar el primer semestre, cuando todavía hay tiempo de intervenir.
9. **Preprocesamiento:** el *pipeline* sin fuga de datos (límites IQR, escalado y one-hot aprendidos solo con el train) y los pesos de clase fueron suficientes; el ajuste de hiperparámetros dio mejoras modestas (+0,01 a +0,03 de macro-F1), lo que confirma que la calidad de los datos pesa más que el tipo de modelo.

## 5.3 Limitaciones
- **Datos de una sola institución y periodo**: no se garantiza que el modelo generalice a otras instituciones o países.
- **Variables macroeconómicas** (`gdp`, `unemployment_rate`, `inflation_rate`): son constantes por cohorte/periodo; aparecen entre las más importantes y pueden estar actuando como identificador del año de ingreso más que como causa real.
- **Clase `Enrolled` pequeña** (159 casos en el test): sus métricas tienen más incertidumbre que las de las otras clases.
- **Asociación no es causalidad**: que los pagos atrasados predigan la deserción no significa que condonarlos la evite.
- **Sin partición temporal**: no se evaluó entrenar con cohortes antiguas y probar con las recientes.

## 5.4 Consideraciones éticas
El modelo debe usarse para **ofrecer apoyo**, nunca para restringir oportunidades a estudiantes. Variables como género, edad, deuda o beca pueden reflejar condiciones socioeconómicas; se recomienda auditar el desempeño por subgrupos (por ejemplo, tasa de falsos negativos por género) antes de cualquier uso real.

## 5.5 Trabajo futuro
- Ajustar el **umbral de decisión** para priorizar recall en `Dropout` y calibrar las probabilidades.
- Comparar con **SMOTE** o submuestreo frente a `class_weight`.
- Probar otros modelos clásicos (SVM con kernel RBF, XGBoost/LightGBM) y un *ensemble* (Stacking).
- Análisis de equidad por subgrupos y partición temporal.
- Despliegue del modelo final en una aplicación interactiva (Streamlit), pendiente como siguiente entregable.